# Lesson 01 Lab — Run BF16 vs INT4 on Your GPU

This is a real GPU benchmark notebook. **Run All** will load the model twice in isolated Python processes, first as BF16 and then as TorchAO weight-only INT4. It measures model memory, Prefill latency, generation latency, a small output regression set, and profiler operator evidence.

The checked-in outputs were produced by executing every cell on an NVIDIA RTX 5090. Your GPU and software stack may produce different results.

## 0. Predict before you run

1. Will INT4 reduce active model memory by exactly 75%?
2. Will INT4 make Prefill faster at 128, 512, and 1,024 tokens?
3. Will Decode throughput improve?
4. Which profiler event would convince you that the intended INT4 operator ran?

Write down your prediction, then execute the notebook from top to bottom. A full run can take several minutes.

In [1]:
from pathlib import Path
import contextlib
import io
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')
with contextlib.redirect_stderr(io.StringIO()):
    import torch
    import torchao
    import transformers
from IPython.display import Markdown, display

if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires a CUDA-capable NVIDIA GPU.')

cwd = Path.cwd().resolve()
relative_lesson = Path('chapters/01-mixed-precision-int4/01-precision-formats')
if (cwd / 'support' / 'benchmark.py').exists():
    lesson_dir = cwd
elif (cwd / relative_lesson / 'support' / 'benchmark.py').exists():
    lesson_dir = cwd / relative_lesson
else:
    raise FileNotFoundError('Launch Jupyter from the repository root or this lesson directory.')

model_source = os.environ.get('AI_INFRA_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
local_files_only = os.environ.get('AI_INFRA_LOCAL_FILES_ONLY', '0') == '1'
model_label = Path(model_source).name if local_files_only else model_source
run_dir = lesson_dir / 'outputs' / 'notebook-run'
raw_dir = run_dir / 'raw'
report_dir = run_dir / 'reports'
if run_dir.exists():
    shutil.rmtree(run_dir)
raw_dir.mkdir(parents=True)
report_dir.mkdir(parents=True)

benchmark_env = os.environ.copy()
benchmark_env.update({
    'CH1_MODEL': model_source,
    'CH1_LOCAL_FILES_ONLY': '1' if local_files_only else '0',
    'CH1_OUTPUT_DIR': str(raw_dir),
    'CH1_REPORT_DIR': str(report_dir),
})
gpu_properties = torch.cuda.get_device_properties(0)
display(Markdown(
    f"**GPU:** {torch.cuda.get_device_name(0)} ({gpu_properties.total_memory / 1024**3:.1f} GiB)  \n"
    f"**Compute capability:** {gpu_properties.major}.{gpu_properties.minor}  \n"
    f"**PyTorch / CUDA:** {torch.__version__} / {torch.version.cuda}  \n"
    f"**TorchAO / Transformers:** {torchao.__version__} / {transformers.__version__}  \n"
    f"**Model:** `{model_label}`  \n"
    '**Execution mode:** fresh BF16 and INT4 GPU measurements'
))

**GPU:** NVIDIA GeForce RTX 5090 (31.4 GiB)  
**Compute capability:** 12.0  
**PyTorch / CUDA:** 2.12.0 / 13.0  
**TorchAO / Transformers:** 0.17.0 / 5.12.0  
**Model:** `Qwen2.5-1.5B-Instruct`  
**Execution mode:** fresh BF16 and INT4 GPU measurements

## Theory bridge — a format is not a speed rank

A precision decision has at least five parts: **storage representation, scale or
packing metadata, compute input dtype, accumulation dtype, and the operator that
consumes the tensor**. TF32, for example, normally keeps FP32 storage while
changing eligible matrix-multiply execution; weight-only INT4 changes weight
storage but normally keeps activations and accumulation wider.

The first theoretical prediction is therefore about memory only:

```text
ideal weight bytes = parameter count × nominal bytes per stored weight
```

Real allocated memory adds unquantized layers, group scales, packed-layout
metadata, temporary conversion buffers, activations, cache, workspaces, and
allocator behavior. The notebook records several ledger lines instead of
pretending that nominal bit width equals runtime memory.

### Mechanism at a glance

```mermaid
flowchart LR
  W["model weights"] --> P["packed INT4 codes + scales"]
  A["BF16 activations"] --> K["weight-only linear kernel"]
  P --> K
  K --> O["BF16/FP32 accumulation and output"]
  O --> C["next layer and KV-cache path"]
  K --> E["memory + operator + latency + quality evidence"]
```

### Walk it step by step

1. **Start with the stored object.** Identify which weights are packed to INT4 and which layers remain BF16.
2. **Follow the runtime data path.** Track packed codes, group scales, BF16 activations, accumulation dtype, and any unpack or dequantization work.
3. **Read four evidence axes.** Evaluate memory, operator identity, latency, and quality independently under one frozen workload.
4. **Make a workload-specific decision.** Use INT4 for the tested path only when its capacity benefit and service gates justify the added kernel work.


## 1. Measurement protocol

- Model: Qwen2.5-1.5B-Instruct by default
- Baseline: BF16 weights and compute
- Candidate: TorchAO weight-only INT4, group size 128, BF16 input/compute
- Batch size: 1
- Prefill lengths: 128, 512, and 1,024 tokens
- Prefill: 3 warm-ups and 10 measured repetitions
- Generation: 128-token prompt, 64 new tokens, 2 warm-ups, 5 repetitions
- Quality probe: 20 fixed multiple-choice questions
- Operator check: PyTorch profiler

BF16 and INT4 run in separate processes so one model and allocator state do not contaminate the other measurement.

In [2]:
def run_mode(mode: str) -> tuple[dict, float]:
    start = time.perf_counter()
    completed = subprocess.run(
        [sys.executable, str(lesson_dir / 'support' / 'benchmark.py'), '--mode', mode],
        cwd=lesson_dir,
        env=benchmark_env,
        text=True,
        capture_output=True,
    )
    elapsed = time.perf_counter() - start
    if completed.returncode != 0:
        tail = '\n'.join((completed.stdout + '\n' + completed.stderr).splitlines()[-30:])
        raise RuntimeError(f'{mode} benchmark failed. Last output lines:\n{tail}')
    payload = json.loads((raw_dir / f'{mode}.json').read_text(encoding='utf-8'))
    if payload.get('status') != 'complete':
        raise RuntimeError(f'{mode} benchmark did not complete')
    return payload, elapsed

print('Benchmark runner ready. The next two cells perform real GPU work.')

Benchmark runner ready. The next two cells perform real GPU work.


## Theory bridge — why the BF16 baseline comes first

The baseline freezes the checkpoint, prompt tokens, batch, sequence lengths,
warm-up, repetitions, decoding policy, and quality probe. BF16 is not assumed
perfect; it is the control needed to attribute any later change to the INT4
candidate. Loading it in a separate process also prevents the candidate's
packing and allocator state from contaminating baseline memory.

## 2. Run the BF16 baseline

In [3]:
bf16, bf16_wall_s = run_mode('bf16')
bf16_prefill = bf16['prefill']
display(Markdown(
    f"**BF16 run completed in {bf16_wall_s:.1f} s**  \n"
    f"CUDA allocated after load: **{bf16['load']['allocated_after_load_gib']:.3f} GiB**  \n"
    f"Unique tensor storage: **{bf16['load']['tensor_storage']['unique_storage_gib']:.3f} GiB**  \n"
    f"Prefill median (128 / 512 / 1024): **{bf16_prefill['128']['median_ms']:.3f} / {bf16_prefill['512']['median_ms']:.3f} / {bf16_prefill['1024']['median_ms']:.3f} ms**  \n"
    f"Approx. Decode throughput: **{bf16['generate']['approximate_decode_tokens_per_s']:.3f} tok/s**"
))

**BF16 run completed in 12.5 s**  
CUDA allocated after load: **2.876 GiB**  
Unique tensor storage: **2.875 GiB**  
Prefill median (128 / 512 / 1024): **9.944 / 11.318 / 19.783 ms**  
Approx. Decode throughput: **101.077 tok/s**

## Theory bridge — what weight-only INT4 actually executes

For each quantized weight group, a reference view is
`q = clamp(round(w / s), -8, 7)` and `w_hat = s·q`. A production weight-only
operator reads packed codes and scales while BF16 activations enter the linear
layer. Smaller storage can reduce memory traffic, but scale loads, unpack or
dequantization, unsupported layers, small-batch shapes, and kernel launch
overhead can outweigh that saving.

This is why the candidate must prove three independent facts: modules were
converted, an INT4 operator appeared in the profiler, and measured latency
changed under the same workload.

## 3. Run the TorchAO INT4 candidate

This cell reloads the same checkpoint, packs eligible linear layers as weight-only INT4, and runs the same protocol.

In [4]:
int4, int4_wall_s = run_mode('int4')
int4_prefill = int4['prefill']
display(Markdown(
    f"**INT4 run completed in {int4_wall_s:.1f} s**  \n"
    f"CUDA allocated after load: **{int4['load']['allocated_after_load_gib']:.3f} GiB**  \n"
    f"Unique tensor storage: **{int4['load']['tensor_storage']['unique_storage_gib']:.3f} GiB**  \n"
    f"Quantized modules: **{int4['load']['int4_module_count']}**  \n"
    f"Prefill median (128 / 512 / 1024): **{int4_prefill['128']['median_ms']:.3f} / {int4_prefill['512']['median_ms']:.3f} / {int4_prefill['1024']['median_ms']:.3f} ms**  \n"
    f"Approx. Decode throughput: **{int4['generate']['approximate_decode_tokens_per_s']:.3f} tok/s**"
))

**INT4 run completed in 14.1 s**  
CUDA allocated after load: **1.336 GiB**  
Unique tensor storage: **1.319 GiB**  
Quantized modules: **113**  
Prefill median (128 / 512 / 1024): **13.619 / 43.963 / 85.478 ms**  
Approx. Decode throughput: **96.966 tok/s**

## Theory bridge — read four evidence axes separately

The comparison is not one winner column. It asks:

1. **Storage/runtime memory:** did stable active bytes and peak bytes fall?
2. **Operator path:** did the intended packed INT4 operation execute?
3. **Performance:** did Prefill and approximate Decode improve at the tested shapes?
4. **Quality:** did outputs remain within the small frozen regression probe?

A pass on one axis does not imply a pass on another. In particular, a real INT4
kernel can execute and save memory while still losing latency.

## 4. Build the comparison from this run

In [5]:
completed = subprocess.run(
    [sys.executable, str(lesson_dir / 'support' / 'summarize.py')],
    cwd=lesson_dir,
    env=benchmark_env,
    text=True,
    capture_output=True,
)
if completed.returncode != 0:
    raise RuntimeError('Result summarization failed')
comparison = json.loads((report_dir / 'comparison.json').read_text(encoding='utf-8'))

memory = comparison['memory']
memory_table = [
    '| Memory account | BF16 | INT4 | INT4 reduction |',
    '|---|---:|---:|---:|',
    f"| Unique tensor storage | {memory['bf16_storage_gib']:.3f} GiB | {memory['int4_storage_gib']:.3f} GiB | {memory['storage_reduction_pct']:.2f}% |",
    f"| CUDA allocated after load | {memory['bf16_allocated_after_load_gib']:.3f} GiB | {memory['int4_allocated_after_load_gib']:.3f} GiB | {memory['allocated_after_load_reduction_pct']:.2f}% |",
    f"| Runtime peak allocated | {memory['bf16_peak_runtime_allocated_gib']:.3f} GiB | {memory['int4_peak_runtime_allocated_gib']:.3f} GiB | — |",
]
display(Markdown('### Memory\n\n' + '\n'.join(memory_table)))

prefill_table = ['| Input tokens | BF16 median | INT4 median | INT4 latency change |', '|---:|---:|---:|---:|']
for tokens, values in comparison['prefill'].items():
    prefill_table.append(
        f"| {tokens} | {values['bf16_median_ms']:.3f} ms | {values['int4_median_ms']:.3f} ms | {values['int4_latency_change_pct']:+.2f}% |"
    )
display(Markdown('### Prefill\n\n' + '\n'.join(prefill_table)))

### Memory

| Memory account | BF16 | INT4 | INT4 reduction |
|---|---:|---:|---:|
| Unique tensor storage | 2.875 GiB | 1.319 GiB | 54.12% |
| CUDA allocated after load | 2.876 GiB | 1.336 GiB | 53.54% |
| Runtime peak allocated | 3.228 GiB | 1.688 GiB | — |

### Prefill

| Input tokens | BF16 median | INT4 median | INT4 latency change |
|---:|---:|---:|---:|
| 128 | 9.944 ms | 13.619 ms | +36.97% |
| 512 | 11.318 ms | 43.963 ms | +288.42% |
| 1024 | 19.783 ms | 85.478 ms | +332.08% |

## 5. Inspect generation, quality, and operator evidence

In [6]:
generation = comparison['generation']
quality = comparison['quality']
operators = comparison['operator_evidence']
display(Markdown(
    '| Check | BF16 | INT4 / evidence |\n'
    '|---|---:|---:|\n'
    f"| Total generation median | {generation['bf16_total_median_ms']:.3f} ms | {generation['int4_total_median_ms']:.3f} ms |\n"
    f"| Approx. Decode throughput | {generation['bf16_approx_decode_tokens_per_s']:.3f} tok/s | {generation['int4_approx_decode_tokens_per_s']:.3f} tok/s |\n"
    f"| Fixed-question accuracy | {quality['bf16_accuracy']:.0%} | {quality['int4_accuracy']:.0%} |\n"
    f"| Quantized module count | {operators['bf16_int4_module_count']} | {operators['int4_module_count']} |\n"
    f"| INT4 profiler event | — | `{', '.join(operators['profiler_int4_event_keys'])}` |"
))

| Check | BF16 | INT4 / evidence |
|---|---:|---:|
| Total generation median | 643.123 ms | 673.642 ms |
| Approx. Decode throughput | 101.077 tok/s | 96.966 tok/s |
| Fixed-question accuracy | 90% | 85% |
| Quantized module count | 0 | 113 |
| INT4 profiler event | — | `aten::_weight_int4pack_mm` |

## Theory bridge — the decision and its reversal conditions

Keep INT4 as the default only if the required capacity gain is real, the native
path executes, quality stays within the predeclared gate, and the relevant
Prefill/Decode or service SLO improves. Otherwise keep BF16 or use INT4 only as
a capacity fallback. A different backend, offline-packed checkpoint, model
shape, batch, or software release is a valid reason to rerun—not a reason to
generalize this result away.

## 6. Make a bounded decision

Do not generalize one backend and one set of shapes to every INT4 implementation or GPU. State what this run established.

In [7]:
memory_win = memory['int4_allocated_after_load_gib'] < memory['bf16_allocated_after_load_gib']
prefill_win = all(row['int4_latency_change_pct'] < 0 for row in comparison['prefill'].values())
decode_win = generation['int4_approx_decode_tokens_per_s'] > generation['bf16_approx_decode_tokens_per_s']
operator_verified = bool(operators['profiler_int4_event_keys'])
decision = (
    'Use this INT4 path as a memory-capacity option, not the default performance path.'
    if memory_win and operator_verified and not (prefill_win and decode_win)
    else 'Re-evaluate the baseline and candidate for this GPU and workload.'
)
display(Markdown(
    f"Real INT4 operator observed: **{operator_verified}**  \n"
    f"Stable active memory decreased: **{memory_win}**  \n"
    f"Prefill became faster at every tested length: **{prefill_win}**  \n"
    f"Approximate Decode throughput increased: **{decode_win}**  \n"
    f"\n**Decision for this run:** {decision}"
))
print('Detailed generated result: outputs/notebook-run/reports/comparison.json')

Real INT4 operator observed: **True**  
Stable active memory decreased: **True**  
Prefill became faster at every tested length: **False**  
Approximate Decode throughput increased: **False**  

**Decision for this run:** Use this INT4 path as a memory-capacity option, not the default performance path.

Detailed generated result: outputs/notebook-run/reports/comparison.json


## Run it with your own model or GPU

By default, the notebook downloads `Qwen/Qwen2.5-1.5B-Instruct`. To use an existing checkpoint, start Jupyter with:

```bash
AI_INFRA_MODEL=/path/to/model AI_INFRA_LOCAL_FILES_ONLY=1 jupyter lab
```

Install the tested GPU dependencies from the repository root with `pip install -r requirements.txt`, and install Jupyter with `pip install -r requirements-notebook.txt`. The current TorchAO INT4 kernel may not support every GPU generation or software version; an unsupported environment should fail explicitly rather than silently report a fake INT4 result.

The 20-question quality set is a regression probe, not a general model-quality benchmark. Decode is approximated by subtracting a separately measured Prefill median from total generation time.